# 10 · Dateien lesen und schreiben

**Lernziele**

- Du arbeitest mit Pfaden über `pathlib`.
- Du liest und schreibst Textdateien mit `with open(...)`.
- Du verarbeitest CSV-Dateien mit dem Modul `csv`.
- Du speicherst und lädst Daten als JSON.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 7 und 14.

## Pfade mit `pathlib`

`Path` beschreibt einen Datei- oder Ordnerpfad, unabhängig vom Betriebssystem. Mit `/` setzt du Pfade zusammen.

In [ ]:
from pathlib import Path

data_dir = Path("../daten")
shopping_list = data_dir / "einkaufsliste.txt"

print(shopping_list)
print(shopping_list.exists())
print(shopping_list.name, shopping_list.suffix)
print(shopping_list.resolve())  # absoluter Pfad

Ordnerinhalt auflisten:

In [ ]:
for path in sorted(data_dir.iterdir()):
    kind = "[DIR]" if path.is_dir() else f"{path.stat().st_size} Bytes"
    print(f"{path.name:<25} {kind}")

**Übung 1 (Datei nicht gefunden, leicht):** Code, den eine KI geschrieben haben könnte (präpariert): `open("daten/faust.txt")` wirft `FileNotFoundError`, obwohl die Datei existiert. Finde mit `Path.cwd()` heraus, wo das Notebook sucht, und baue den Pfad so, dass `exists()` `True` liefert.

In [ ]:
with open("daten/faust.txt", encoding="utf-8") as file:
    print(file.readline())

In [ ]:
print(Path.cwd())  # das Notebook läuft im Ordner loesungen/
faust = Path("../daten/faust.txt")
print(faust.exists(), faust.resolve())
# Hinweis für den Trainer: Eine Chat-KI kennt den Arbeitsordner des Notebooks nicht,
# genau daher kommt der Fehler oft.

## Textdateien lesen

`with open(...)` öffnet die Datei und schließt sie am Ende des Blocks automatisch, auch bei Fehlern. Gib immer `encoding="utf-8"` an. Seit Python 3.15 ist UTF-8 zwar überall die Voreinstellung, ältere Versionen nehmen unter Windows aber eine andere Kodierung, und dann gehen Umlaute kaputt. *Explicit is better than implicit:* Mit `encoding="utf-8"` läuft dein Code mit jeder Version gleich.

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    content = file.read()
print(content)

Zeile für Zeile lesen ist bei großen Dateien sparsamer:

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line.strip()}")

Für kleine Dateien gibt es eine Abkürzung:

In [ ]:
shopping_list.read_text(encoding="utf-8").splitlines()

## Textdateien schreiben

Der **Modus** bestimmt, was passiert:

| Modus | Bedeutung |
| --- | --- |
| `"r"` | lesen (Standard) |
| `"w"` | schreiben, **überschreibt** vorhandenen Inhalt |
| `"a"` | anhängen |

In [ ]:
output_dir = Path("../daten/ausgabe")
output_dir.mkdir(exist_ok=True)
notes = output_dir / "notizen.txt"

with open(notes, "w", encoding="utf-8") as file:
    file.write("Erste Zeile\n")
    file.write("Zweite Zeile\n")

with open(notes, "a", encoding="utf-8") as file:
    file.write("Angehängte Zeile\n")

print(notes.read_text(encoding="utf-8"))

**Übung 2 (Nur die letzte Notiz bleibt, leicht):** Code, den eine KI geschrieben haben könnte (präpariert, ein Fehler eingebaut): Er soll drei Erinnerungen in `erinnerungen.txt` schreiben, in der Datei steht danach aber nur die letzte. Finde den Fehler und behebe ihn.

In [ ]:
notes = ["Milch kaufen", "Zahnarzt anrufen", "Steuer abgeben"]
target = Path("../daten/ausgabe/erinnerungen.txt")
for note in notes:
    with open(target, "w", encoding="utf-8") as file:
        file.write(note + "\n")
print(target.read_text(encoding="utf-8"))

In [ ]:
notes = ["Milch kaufen", "Zahnarzt anrufen", "Steuer abgeben"]
target = Path("../daten/ausgabe/erinnerungen.txt")
with open(target, "w", encoding="utf-8") as file:  # einmal öffnen, nicht pro Notiz
    for note in notes:
        file.write(note + "\n")
print(target.read_text(encoding="utf-8"))

## CSV-Dateien

CSV ist das Austauschformat für Tabellen, etwa aus Excel. Das Modul `csv` kümmert sich um Trennzeichen und Anführungszeichen. `DictReader` liefert jede Zeile als Dictionary mit den Spaltennamen als Schlüssel.

In [ ]:
import csv

with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        print(f"{row['name']:<10} {row['points']:>5} Punkte, Level {row['level']}")

> Werte aus CSV sind immer Strings. Zum Rechnen umwandeln: `int(row["points"])`.

Schreiben mit `DictWriter`:

In [ ]:
results = [
    {"name": "Ada", "points": 1200},
    {"name": "Linus", "points": 950},
]

with open(output_dir / "ergebnisse.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "points"])
    writer.writeheader()
    writer.writerows(results)

print((output_dir / "ergebnisse.csv").read_text(encoding="utf-8"))

## JSON

JSON speichert verschachtelte Daten aus Listen und Dictionaries. Es ist das Standardformat von Web-APIs.

In [ ]:
import json

with open(data_dir / "personen.json", encoding="utf-8") as file:
    data = json.load(file)

for person in data["people"]:
    print(f"{person['name']} aus {person['city']}")

In [ ]:
data["people"].append({"name": "Grace", "city": "New York"})

with open(output_dir / "personen_neu.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=2, ensure_ascii=False)

print((output_dir / "personen_neu.json").read_text(encoding="utf-8"))

**Übung 3 (JSON-Rundreise, leicht):** Speichere die Einstellungen aus `settings` mit `json.dump` in die Datei `../daten/ausgabe/einstellungen.json`. Lies die Datei danach mit `json.load` wieder ein und prüfe mit `==`, ob du dasselbe Dictionary zurückbekommst.

Untersuchen: Mach aus der Liste `recent_files` ein Tupel und wiederhole die Rundreise. Ist das Geladene noch gleich dem Original, und warum? Was passiert, wenn `recent_files` ein Set ist?

In [ ]:
settings = {
    "user": "Jörg",
    "font_size": 14,
    "dark_mode": True,
    "recent_files": ["bericht.docx", "umsatz.csv"],
}

In [ ]:
settings_file = output_dir / "einstellungen.json"

with open(settings_file, "w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2, ensure_ascii=False)

with open(settings_file, encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded)
print(loaded == settings)  # True

# Untersuchen: JSON kennt nur dict, list, str, int, float, bool und None.
# Ein Tupel speichert json.dump als Liste, zurück kommt also eine Liste.
with_tuple = {**settings, "recent_files": ("bericht.docx", "umsatz.csv")}
with open(settings_file, "w", encoding="utf-8") as file:
    json.dump(with_tuple, file, indent=2, ensure_ascii=False)
with open(settings_file, encoding="utf-8") as file:
    loaded = json.load(file)
print(loaded["recent_files"], loaded == with_tuple)  # Liste, False

# Ein Set kennt JSON gar nicht: TypeError (Zelle darunter).
# Abhilfe: das Set vor dem Speichern mit sorted(...) in eine Liste umwandeln.

In [ ]:
with_set = {**settings, "recent_files": {"bericht.docx", "umsatz.csv"}}
json.dumps(with_set)  # wie json.dump, liefert den Text aber ohne Datei: kein halb geschriebenes JSON

## Übungen

Die Übungen 4 bis 6 arbeiten mit Beispieldateien. Die jeweilige Vorbereitungszelle legt sie in `../daten/ausgabe/` an; du kannst sie beliebig oft ausführen.

**Übung 4 (Logdatei nach Fehlern durchsuchen, mittel):** Lies `../daten/ausgabe/beispieldaten/server.log` (die Zelle darunter erzeugt sie), zähle die Zeilen je Level und schreibe alle `ERROR`-Zeilen mit Zeilennummer nach `../daten/ausgabe/fehler.txt`. Öffne beide Dateien in einem `with`.

Tipp: `from collections import Counter`; ein `Counter` zählt gleiche Einträge (wie in Notebook 06 bei den Logzeilen), `.most_common()` sortiert nach Häufigkeit.

In [ ]:
# Vorbereitung: erzeugt eine Beispiel-Logdatei (Datum, Uhrzeit, Level, Meldung)
sample_dir = Path("../daten/ausgabe/beispieldaten")
sample_dir.mkdir(parents=True, exist_ok=True)
log_text = """\
2026-11-02 08:58:01 INFO Server gestartet auf Port 8080
2026-11-02 08:58:02 INFO Datenbankverbindung hergestellt
2026-11-02 09:00:12 INFO Anmeldung: user=anna
2026-11-02 09:01:45 INFO Bericht erstellt: umsatz_oktober.pdf
2026-11-02 09:03:10 WARNING Langsame Antwort: /kunden (2.4 s)
2026-11-02 09:05:33 INFO Anmeldung: user=ben
2026-11-02 09:07:02 ERROR Datei nicht gefunden: rechnung_4711.pdf
2026-11-02 09:07:03 INFO Anmeldung: user=chris
2026-11-02 09:10:27 WARNING Speicherplatz unter 15 %
2026-11-02 09:12:48 INFO Bericht erstellt: kunden.csv
2026-11-02 09:15:00 INFO Sicherung gestartet
2026-11-02 09:18:21 ERROR Sicherung abgebrochen: Zielordner nicht erreichbar
2026-11-02 09:18:22 INFO Sicherung wird in 5 Minuten wiederholt
2026-11-02 09:20:05 INFO Abmeldung: user=anna
2026-11-02 09:23:22 INFO Sicherung gestartet
2026-11-02 09:26:40 INFO Sicherung abgeschlossen
2026-11-02 09:30:11 WARNING Falsches Passwort: user=dora
2026-11-02 09:30:19 WARNING Falsches Passwort: user=dora
2026-11-02 09:30:25 ERROR Konto gesperrt nach 3 Fehlversuchen: user=dora
2026-11-02 09:35:50 INFO Anmeldung: user=emil
2026-11-02 09:41:02 INFO Bericht erstellt: lager.xlsx
2026-11-02 09:44:17 WARNING Langsame Antwort: /lager (3.1 s)
2026-11-02 09:48:39 INFO Abmeldung: user=ben
2026-11-02 09:52:08 ERROR Datenbankverbindung verloren
2026-11-02 09:52:10 INFO Datenbankverbindung hergestellt
2026-11-02 09:55:31 INFO Abmeldung: user=chris
2026-11-02 10:00:00 INFO Stündlicher Statusbericht: alles in Ordnung
2026-11-02 10:02:44 INFO Anmeldung: user=anna
2026-11-02 10:05:16 WARNING Zertifikat läuft in 14 Tagen ab
2026-11-02 10:09:58 ERROR Mailversand fehlgeschlagen: smtp.example.org
"""
(sample_dir / "server.log").write_text(log_text, encoding="utf-8")

In [ ]:
from collections import Counter

log = Path("../daten/ausgabe/beispieldaten/server.log")
errors = Path("../daten/ausgabe/fehler.txt")
levels = Counter()
with open(log, encoding="utf-8") as source, open(errors, "w", encoding="utf-8") as target:
    for number, line in enumerate(source, start=1):
        day, clock, level, message = line.split(maxsplit=3)
        levels[level] += 1
        if level == "ERROR":
            target.write(f"{number}: {line}")  # line endet schon mit "\n"
print(levels.most_common())
print(errors.read_text(encoding="utf-8"))

**Übung 5 (Kontoauszug: Annahmen brechen, mittel):** Der Startcode (wie ihn eine KI für „Beträge aus dem Online-Banking summieren“ schreiben könnte) läuft mit den Beispielwerten. Welche Annahmen trifft er über die Beträge? Schreibe für jede Annahme einen Wert, der sie verletzt, und prüfe ihn. Dann werte `../daten/ausgabe/beispieldaten/kontoauszug.csv` (die Vorbereitungszelle erzeugt sie: Semikolon, Dezimalkomma, Tausenderpunkt, BOM) robust aus: Einnahmen, Ausgaben, Saldo und Ausgaben je Empfänger, absteigend.

- Untersuchen: Was passiert mit `"2.500,00"` (Tausenderpunkt) und mit `""` (leerer Betrag)? Bricht eine leere Zeile am Ende der CSV-Datei die Auswertung?
- Ändern: Entferne den Tausenderpunkt, überspringe Zeilen ohne Betrag und werte dann die ganze Datei aus.

In [ ]:
from decimal import Decimal

amounts = ["-850,00", "-62,35", "1200,00"]  # aus dem Kontoauszug
total = Decimal(0)
for amount in amounts:
    total += Decimal(amount.replace(",", "."))
print("Saldo:", total, "€")

In [ ]:
# Vorbereitung: erzeugt einen Kontoauszug, wie ihn deutsche Banken exportieren
# (Semikolon, Dezimalkomma, Tausenderpunkt, BOM, eine Zeile ";;;", Leerzeile am Ende).
sample_dir = Path("../daten/ausgabe/beispieldaten")
sample_dir.mkdir(parents=True, exist_ok=True)
statement_text = """\
Datum;Empfänger;Verwendungszweck;Betrag
01.10.2026;Arbeitgeber GmbH;Gehalt Oktober;2.500,00
01.10.2026;Hausverwaltung Meier;Miete Oktober;-850,00
02.10.2026;Stadtwerke;Abschlag Strom;-62,35
05.10.2026;Supermarkt;Einkauf;-45,10
12.10.2026;Supermarkt;Einkauf;-38,90
15.10.2026;Fitnessstudio;Beitrag;-29,90
20.10.2026;Hausverwaltung Meier;Nebenkosten 2025;-1.120,00
;;;
28.10.2026;Finanzamt;Erstattung 2025;312,40

"""
(sample_dir / "kontoauszug.csv").write_text(statement_text, encoding="utf-8-sig")

In [ ]:
from decimal import InvalidOperation

# Annahmen des Startcodes: Komma als Dezimaltrenner, kein Tausenderpunkt, nie leer.
for breaking in ["2.500,00", ""]:
    try:
        Decimal(breaking.replace(",", "."))
    except InvalidOperation:
        print(f"bricht den Startcode: {breaking!r}")

# Eine leere Zeile am Dateiende überspringt csv.DictReader; die Zeile ";;;" dagegen
# liefert einen leeren Betrag und bricht die Auswertung ohne eigene Prüfung.
statement = Path("../daten/ausgabe/beispieldaten/kontoauszug.csv")
with open(statement, encoding="utf-8-sig", newline="") as file:
    rows = list(csv.DictReader(file, delimiter=";"))
income, spending, by_payee = Decimal(0), Decimal(0), {}
for row in rows:
    if not row["Betrag"]:
        continue  # Zeile ohne Betrag, z. B. ";;;"
    amount = Decimal(row["Betrag"].replace(".", "").replace(",", "."))
    if amount >= 0:
        income += amount
    else:
        spending -= amount
        by_payee[row["Empfänger"]] = by_payee.get(row["Empfänger"], Decimal(0)) - amount
print(f"Einnahmen {income} €, Ausgaben {spending} €, Saldo {income - spending} €")
for payee in sorted(by_payee, key=by_payee.get, reverse=True):
    print(f"{payee:<20} {by_payee[payee]:>10} €")

**Übung 6 (Download-Ordner aufräumen, schwer):** Sortiere alle Dateien in `../daten/ausgabe/downloads` nach Endung in Unterordner (`pdf`, `jpg`, …). Gib zuerst nur aus, was passieren würde (`DRY_RUN = True`), und verschiebe erst danach mit `Path.rename`. Eine Datei, die es am Ziel schon gibt, wird nicht überschrieben, sondern gemeldet. Die Zelle darunter legt den Ordner mit Beispieldateien an.

In [ ]:
import shutil

# Vorbereitung: legt den Beispielordner neu an. Gelöscht wird nur dieser Übungsordner.
downloads = Path("../daten/ausgabe/downloads")
shutil.rmtree(downloads, ignore_errors=True)
(downloads / "pdf").mkdir(parents=True)
(downloads / "alt").mkdir()
for name in [
    "rechnung_oktober.pdf",
    "vertrag.pdf",
    "urlaub_01.JPG",
    "urlaub_02.jpg",
    "einkaufsliste.txt",
    "haushaltsbuch.xlsx",
    "README",
]:
    (downloads / name).write_text(f"Inhalt von {name}\n", encoding="utf-8")
(downloads / "pdf" / "vertrag.pdf").write_text("ältere Fassung\n", encoding="utf-8")
(downloads / "alt" / "notiz.txt").write_text("alte Notiz\n", encoding="utf-8")
print(sorted(path.name for path in downloads.iterdir()))

In [ ]:
DRY_RUN = True  # erst prüfen, dann auf False setzen und erneut ausführen
downloads = Path("../daten/ausgabe/downloads")
for path in sorted(downloads.iterdir()):
    if not path.is_file():
        continue
    target = downloads / (path.suffix.lower().removeprefix(".") or "ohne_endung") / path.name
    if target.exists():
        print(f"übersprungen, existiert schon: {target}")
    elif DRY_RUN:
        print(f"würde verschieben: {path.name} -> {target.parent.name}/")
    else:
        target.parent.mkdir(exist_ok=True)
        path.rename(target)

## Zusammenfassung

- `Path("ordner") / "datei.txt"` baut Pfade, `.exists()`, `.iterdir()`, `.mkdir()` helfen beim Umgang.
- Immer `with open(pfad, modus, encoding="utf-8") as file:`.
- Modus `"r"` lesen, `"w"` überschreiben, `"a"` anhängen.
- CSV: `csv.DictReader` und `csv.DictWriter`, dabei `newline=""` angeben.
- JSON: `json.load` / `json.dump(..., indent=2, ensure_ascii=False)`.